In [1]:
!pip install pennylane datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 37.5 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import pennylane as qml
from pennylane import numpy as np

# Charger le dataset
ds = load_dataset("Genius-Society/Pima")

# Extraire X et y
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'],
               s['SkinThickness'], s['Insulin'], s['BMI'],
               s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']], dtype=float)

y = np.array([s['Outcome'] for s in ds['train']])


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [3]:
def normalize(x):
    return x / np.linalg.norm(x)




In [4]:
def feature_matrix(x):
    x = normalize(x)
    return np.diag(x)



In [5]:
n_data = 8
n_ancilla = 3
n_qubits = n_data + n_ancilla

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def block_encoding_expectation(x):
    x = normalize(x)

    # Ancillas en superposition
    for i in range(n_ancilla):
        qml.Hadamard(wires=i)

    # Block encoding (simplifié)
    for i in range(n_data):
        qml.ctrl(qml.RY, control=range(n_ancilla))(x[i], wires=n_ancilla + i)

    # 🔑 OUTPUT TESTABLE
    return qml.expval(qml.PauliZ(wires=n_ancilla))





In [6]:
out = block_encoding_expectation(X[0])
print(out)


0.9999838411319012


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Préparation des données
# On convertit en réel car les arbres de décision ne supportent pas les nombres complexes
X_norm = np.array([normalize(x) for x in X])
all_outputs = np.array([block_encoding_expectation(x) for x in X_norm])
X_final = all_outputs.reshape(-1, 1) # On reformate pour l'algorithme

# D. L'Arbre de Décision
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# E. Résultat
print(f"Accuracy Block Encoding: {clf.score(X_test, y_test):.2%}")

Accuracy Block Encoding: 64.23%
